# NHL Draft Analysis

This notebook loads the draft scores produced by formula.ipynb, applies 
franchise consolidation, adds team colors, and exports clean CSVs for 
the Tableau dashboard.

## Setup

In [1]:
import pandas as pd

df = pd.read_csv('../data/outputs/draft_scores_adjusted.csv')
print(f"Loaded {len(df)} rows")
df.head()

Loaded 2484 rows


,draft_year,round,pick_number,drafted_by,player_name,position,games_played,points,performance_score,hindsight_rank,plus_minus
0,2005,1,1,Pittsburgh,Sidney Crosby,C,1408.0,1746.0,5406.34,1,0
1,2005,1,2,Anaheim,Bobby Ryan,R,866.0,569.0,2169.01,9,-7
2,2005,1,3,Carolina,Jack Johnson,D,1228.0,342.0,2011.18,12,-9
3,2005,1,4,Minnesota,Benoit Pouliot,L,625.0,263.0,1227.27,21,-17
4,2005,1,6,Columbus,Gilbert Brule,C,299.0,95.0,516.55,39,-33


## Team Rankings

Overall team draft rankings from 2005 to 2017, measured by cumulative 
plus/minus across all picks and all drafts.

In [2]:
team_rankings = df.groupby('drafted_by').agg(
    total_plus_minus=('plus_minus', 'sum'),
    total_picks=('plus_minus', 'count'),
    avg_plus_minus_per_pick=('plus_minus', 'mean')
).round(2).reset_index()

team_rankings = team_rankings.sort_values('total_plus_minus', ascending=False).reset_index(drop=True)
team_rankings.index += 1
team_rankings.columns = ['team', 'total_plus_minus', 'total_picks', 'avg_per_pick']

print("NHL Team Draft Rankings 2005-2017")
print(team_rankings.to_string())

NHL Team Draft Rankings 2005-2017
            team  total_plus_minus  total_picks  avg_per_pick
1       San Jose              2088           81         25.78
2    Los Angeles              1971           88         22.40
3        Toronto              1631           87         18.75
4      Tampa Bay              1488           86         17.30
5   Philadelphia              1400           78         17.95
6         Ottawa              1313           78         16.83
7     Washington              1217           78         15.60
8       Columbus              1179           85         13.87
9      Nashville              1023           85         12.04
10    New Jersey               986           84         11.74
11        Dallas               931           75         12.41
12    Pittsburgh               927           71         13.06
13       Detroit               913           87         10.49
14  NY Islanders               897           88         10.19
15        Boston               893  

## Consistency Analysis

Team plus/minus broken down by draft year. Reveals whether a team's 
overall ranking is driven by consistent performance or a few standout 
draft classes.

In [3]:
team_by_year = df.groupby(['drafted_by', 'draft_year']).agg(
    plus_minus=('plus_minus', 'sum'),
    total_picks=('plus_minus', 'count'),
    avg_plus_minus_per_pick=('plus_minus', 'mean')
).round(2).reset_index()

team_by_year = team_by_year.rename(columns={'drafted_by': 'team'})

print(f"team_by_year shape: {team_by_year.shape}")
team_by_year.head(20)

team_by_year shape: (391, 5)


,team,draft_year,plus_minus,total_picks,avg_plus_minus_per_pick
0,Anaheim,2005,-20,5,-4.00
1,Anaheim,2006,85,5,17.00
2,Anaheim,2007,-41,6,-6.83
3,Anaheim,2008,-57,9,-6.33
4,Anaheim,2009,-12,6,-2.00
5,Anaheim,2010,128,8,16.00
6,Anaheim,2011,205,6,34.17
7,Anaheim,2012,115,7,16.43
8,Anaheim,2013,-46,5,-9.20
9,Anaheim,2014,187,5,37.40


## Round Analysis

Team plus/minus broken down by draft round. Reveals whether certain 
teams excel at finding late round steals or consistently capitalize 
on early round picks.

In [4]:
team_by_round = df.groupby(['drafted_by', 'round']).agg(
    plus_minus=('plus_minus', 'sum'),
    total_picks=('plus_minus', 'count'),
    avg_plus_minus_per_pick=('plus_minus', 'mean')
).round(2).reset_index()

team_by_round = team_by_round.rename(columns={'drafted_by': 'team'})

print(f"team_by_round shape: {team_by_round.shape}")
team_by_round.head(20)
print(team_by_round.to_string())

team_by_round shape: (216, 5)
             team  round  plus_minus  total_picks  avg_plus_minus_per_pick
0         Anaheim      1        -232           15                   -15.47
1         Anaheim      2        -270           16                   -16.88
2         Anaheim      3         -49           12                    -4.08
3         Anaheim      4         207           10                    20.70
4         Anaheim      5         354           12                    29.50
5         Anaheim      6         234            7                    33.43
6         Anaheim      7         314            6                    52.33
7         Arizona      1        -402           18                   -22.33
8         Arizona      2        -294           13                   -22.62
9         Arizona      3        -117           14                    -8.36
10        Arizona      4         201            9                    22.33
11        Arizona      5         261           11                    2

The round breakdown reveals a consistent pattern across all teams — rounds 
1 through 3 are almost universally negative, while rounds 4 through 7 
produce increasingly large positive scores. This confirms the late round 
asymmetry identified in formula.ipynb: the metric rewards late round steals 
more than it penalizes early round busts, and the cumulative team rankings 
are heavily influenced by late round performance.

To supplement the main ranking, we produce a separate dataset restricted 
to rounds 1 through 3 — the picks where scouting skill is most scrutinized 
and where the late round asymmetry does not apply.

### Early Rounds Ranking — Rounds 1-3 Only

In [5]:
df_early = df[df['round'] <= 3]

team_rankings_early = df_early.groupby('drafted_by').agg(
    total_plus_minus=('plus_minus', 'sum'),
    total_picks=('plus_minus', 'count'),
    avg_plus_minus_per_pick=('plus_minus', 'mean')
).round(2).reset_index()

team_rankings_early = team_rankings_early.sort_values('total_plus_minus', ascending=False).reset_index(drop=True)
team_rankings_early.index += 1
team_rankings_early = team_rankings_early.rename(columns={'drafted_by': 'team'})

print("NHL Team Draft Rankings 2005-2017 — Rounds 1-3 Only")
print(team_rankings_early.to_string())

NHL Team Draft Rankings 2005-2017 — Rounds 1-3 Only
            team  total_plus_minus  total_picks  avg_plus_minus_per_pick
1          Vegas               -17            6                    -2.83
2        Detroit              -128           37                    -3.46
3    Los Angeles              -129           35                    -3.69
4      Nashville              -135           36                    -3.75
5     New Jersey              -170           37                    -4.59
6     Pittsburgh              -205           28                    -7.32
7   Philadelphia              -251           33                    -7.61
8     Washington              -279           29                    -9.62
9       Carolina              -313           38                    -8.24
10     Minnesota              -331           27                   -12.26
11    NY Rangers              -373           33                   -11.30
12     Tampa Bay              -373           38                    -9.82

## Tableau Preparation

All outputs are saved to data/outputs/ for use in the Tableau dashboard.

In [6]:
# Export all CSVs
team_rankings.to_csv('../data/outputs/team_rankings.csv', index=False)
team_by_year.to_csv('../data/outputs/team_by_year.csv', index=False)
team_by_round.to_csv('../data/outputs/team_by_round.csv', index=False)
team_rankings_early.to_csv('../data/outputs/team_rankings_early.csv', index=False)

print("All files exported successfully!")
print(f"  team_rankings.csv       — {len(team_rankings)} rows")
print(f"  team_by_year.csv        — {len(team_by_year)} rows")
print(f"  team_by_round.csv       — {len(team_by_round)} rows")
print(f"  team_rankings_early.csv — {len(team_rankings_early)} rows")

All files exported successfully!
  team_rankings.csv       — 31 rows
  team_by_year.csv        — 391 rows
  team_by_round.csv       — 216 rows
  team_rankings_early.csv — 31 rows


## End of Analysis

In [7]:
print("Analysis complete.")

Analysis complete.


All datasets have been prepared and exported to data/outputs/. 
These files serve as the data source for the Tableau dashboard.